# 04 - 5-fold cross-validation (both MiniConvNet variants)

**What this notebook does**: runs stratified 5-fold CV on the `faithful` split for both architecture
variants and writes the **canonical MiniConvNet rows** to `outputs/results_table.csv` as
mean +/- std.

**Why this is the headline number (LESSON 8)**: in the earlier attempt a single favourable split
looked far better than the true average. The mean +/- std across folds is what goes in the main
comparison table and the paper-comparison section; single-run numbers from notebook 02 stay in
`experiments_log.csv`.

**What must already exist**: split CSVs from notebook 00. GPU strongly recommended - this is
2 variants x 5 folds = **10 training runs** and is the longest notebook in the project.

**Protocol**
* All `faithful` images (train + val + test folders pooled) are re-partitioned by `StratifiedKFold`.
* Within each fold, 15% of the training portion is held out as a validation set for early stopping,
  so the fold's test portion is never used for model selection.
* Every fold is collapse-checked; collapsed folds are excluded from the mean and reported separately
  (LESSON 3).

**Known caveat, state it whenever you quote these numbers**: the `faithful` data contains duplicate
images, so pooled CV can place copies of the same image in both the training and testing portion of
a fold. That is a property of the paper-comparable protocol, not a bug in this notebook - the
leakage-free counterpart is the `clean` split experiment in notebook 02.

**What "looks right"**: 5 folds per variant, all `ok`, and a std that is small relative to the mean
(a std above ~0.10 means the result is dominated by which split you happened to draw).

In [3]:
import sys
import numpy as np

print("Python:")
print(sys.executable)

print("\nNumPy:")
print(np.__version__)
print(np.__file__)

Python:
/mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/.venv-wsl/bin/python

NumPy:
2.1.3
/mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/.venv-wsl/lib/python3.12/site-packages/numpy/__init__.py


In [4]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.19.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('folds:', CV_FOLDS, '| epochs per fold:', EPOCHS_CV)

data root: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/Data
folds: 5 | epochs per fold: 60


In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split

from src.data_utils import load_split, make_dataset
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, gpu_report, compile_model, make_callbacks,
                            save_history, final_epoch_summary, run_name_for)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                               summarize_cv, format_mean_std, result_row_from_metrics,
                               record_canonical, record_experiment, plot_confusion_matrix,
                               tumor_vs_subtype_breakdown, load_results)

set_global_seeds(SEED)
print(gpu_report())

{'tensorflow': '2.19.1', 'num_gpus': 1, 'gpus': ['/physical_device:GPU:0'], 'note': '0 GPUs on Kaggle means the accelerator is not set to GPU.'}


## 1. Pool the faithful split

**Looks right**: ~1000 rows, class counts matching notebook 00's totals column.

In [8]:
pool = load_split('faithful').reset_index(drop=True)
print('pooled images:', len(pool))
print(pool['class'].value_counts().reindex(CLASS_NAMES))
print()
print('duplicate content hashes in the pool:',
      int((pool['hash'].value_counts() > 1).sum()),
      '- CV folds may therefore share duplicated images (documented caveat).')

FileNotFoundError: 1000 files from the saved split are missing under /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/Data. First missing: C:\Users\shrey\OneDrive\sem7\Medical Image\Project\Lung-Cancer-Classification-\Data\test\adenocarcinoma\000108 (3).png

## 2. Fold definitions

**Looks right**: 5 folds, each test portion ~200 images with all 4 classes represented.

In [ ]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(pool.index.values, pool['label'].values))

for i, (tr_idx, te_idx) in enumerate(folds, start=1):
    te_counts = pool.iloc[te_idx]['class'].value_counts().reindex(CLASS_NAMES).to_dict()
    print(f'fold {i}: train={len(tr_idx):4d} test={len(te_idx):4d} test class counts={te_counts}')

## 3. Single-fold runner

Trains one fold end to end and returns its metrics plus its collapse verdict.

In [ ]:
def run_fold(arch_variant, fold_idx, tr_idx, te_idx, verbose=2):
    set_global_seeds(SEED + fold_idx)   # different init per fold, still reproducible
    run_name = run_name_for('cv', arch_variant, 'faithful', f'fold{fold_idx}')

    train_pool = pool.iloc[tr_idx].reset_index(drop=True)
    test_df = pool.iloc[te_idx].reset_index(drop=True)
    tr_df, val_df = train_test_split(train_pool, test_size=0.15,
                                     stratify=train_pool['class'],
                                     random_state=SEED + fold_idx)

    train_ds = make_dataset(tr_df, shuffle=True, augment=True, seed=SEED + fold_idx)
    val_ds = make_dataset(val_df)
    test_ds = make_dataset(test_df)

    model = build_miniconvnet(arch_variant)
    compile_model(model)                       # Adam(1e-4, clipnorm=1.0)

    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CV,
                        callbacks=make_callbacks(run_name, checkpoint=False),
                        verbose=verbose)
    save_history(history, run_name)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)

    print(f"\nfold {fold_idx} [{arch_variant}] -> acc={metrics['accuracy']:.4f} "
          f"f1={metrics['f1_macro']:.4f} kappa={metrics['cohen_kappa']:.4f} "
          f"status={collapse['status']}")
    if collapse['collapsed']:
        print_collapse_report(collapse, run_name)

    tf.keras.backend.clear_session()
    return {'fold': fold_idx, 'run_name': run_name, 'status': collapse['status'],
            'epochs_trained': final_epoch_summary(history)['epochs_trained'],
            'y_true': y_true, 'y_pred': y_pred, **metrics}

## 4. CV for the `gap` variant (~106K params)

This cell trains 5 models. Expect it to take a while; `verbose=2` prints one line per epoch so you
can see progress and spot a flat loss curve early.

In [ ]:
cv_gap = [run_fold('gap', i, tr, te) for i, (tr, te) in enumerate(folds, start=1)]
print('\ngap folds complete:', len(cv_gap))

## 5. CV for the `flatten` variant (~0.5M params, paper-sized)

In [ ]:
cv_flatten = [run_fold('flatten', i, tr, te) for i, (tr, te) in enumerate(folds, start=1)]
print('\nflatten folds complete:', len(cv_flatten))

## 6. Per-fold tables

**Looks right**: 5 rows per variant, every `status` equal to `ok`. Any collapsed fold is excluded
from the mean below and called out explicitly.

In [ ]:
cols = ['fold', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro',
        'cohen_kappa', 'mcc', 'epochs_trained', 'status']
for name, folds_res in (('gap', cv_gap), ('flatten', cv_flatten)):
    tbl = pd.DataFrame(folds_res)[cols]
    print(f'--- {name} ---')
    print(tbl.round(4).to_string(index=False))
    print()

## 7. Aggregate to mean +/- std and write the canonical rows

`results_table.csv` keeps exactly one row per model (LESSON 7), so re-running this notebook replaces
the MiniConvNet rows rather than appending duplicates.

In [ ]:
def aggregate_and_record(arch_variant, folds_res):
    valid = [f for f in folds_res if f['status'] == VALID_TAG]
    dropped = len(folds_res) - len(valid)
    if not valid:
        raise RuntimeError(f'All folds collapsed for {arch_variant} - nothing valid to report.')

    summary = summarize_cv(valid)
    params = count_params(build_miniconvnet(arch_variant))['total_params']
    tf.keras.backend.clear_session()

    print(f'--- {arch_variant}: {len(valid)}/{len(folds_res)} valid folds '
          f'({dropped} excluded as collapsed) ---')
    for k in ('accuracy', 'f1_macro', 'cohen_kappa', 'mcc'):
        print(f"  {k}: {format_mean_std(summary[k + '_mean'], summary[k + '_std'])}")

    metrics = {k[:-5]: v for k, v in summary.items() if k.endswith('_mean')}
    row = result_row_from_metrics(
        model_name=f'MiniConvNet-{arch_variant} (5-fold CV)',
        metrics=metrics,
        collapse={'status': VALID_TAG},
        arch_variant=arch_variant, split_variant='faithful', params=params,
        accuracy_std=summary['accuracy_std'],
        epochs_trained=int(np.mean([f['epochs_trained'] for f in valid])),
        n_runs=len(valid),
        notes=(f"5-fold CV mean+/-std on the faithful (paper-comparable) split; "
               f"f1_macro={format_mean_std(summary['f1_macro_mean'], summary['f1_macro_std'])}; "
               f"{dropped} fold(s) excluded as collapsed; pooled CV shares duplicated images "
               f"across folds (documented dataset caveat)"))
    path = record_canonical(row)
    print('  written to', path)
    return summary, row

summary_gap, row_gap = aggregate_and_record('gap', cv_gap)
print()
summary_flatten, row_flatten = aggregate_and_record('flatten', cv_flatten)

## 8. Pooled confusion matrix across folds

Concatenating every fold's predictions gives a confusion matrix over the whole dataset, which is
more stable than any single fold's.

**Looks right**: strong `normal` column/row separation, with the residual errors concentrated among
the three tumour subtypes (LESSON 9).

In [ ]:
for name, folds_res in (('gap', cv_gap), ('flatten', cv_flatten)):
    valid = [f for f in folds_res if f['status'] == VALID_TAG]
    y_true = np.concatenate([f['y_true'] for f in valid])
    y_pred = np.concatenate([f['y_pred'] for f in valid])
    print(f'--- {name}: pooled over {len(valid)} folds, {len(y_true)} predictions ---')
    plot_confusion_matrix(y_true, y_pred, f'cv_{name}_faithful_pooled')
    bd = tumor_vs_subtype_breakdown(y_true, y_pred)
    for k, v in bd.items():
        print(f'  {k}: {v}')
    print()

## 9. Also log every individual fold to the experiments log

Keeps the canonical table clean while preserving the per-fold detail (LESSON 7).

In [ ]:
for arch_variant, folds_res in (('gap', cv_gap), ('flatten', cv_flatten)):
    for f in folds_res:
        record_experiment(result_row_from_metrics(
            model_name=f['run_name'], metrics=f,
            collapse={'status': f['status']},
            arch_variant=arch_variant, split_variant='faithful',
            epochs_trained=f['epochs_trained'], n_runs=1,
            config_note=f"CV fold {f['fold']}/{CV_FOLDS} on the faithful split"))

print('canonical table:')
print(load_results('canonical')[['model', 'accuracy', 'accuracy_std', 'n_runs', 'status']].to_string(index=False))
print('\nexperiments_log rows:', len(load_results('experiment')))
print('\nnext: 05_train_baselines.ipynb')